# 03 — Território, Hierarquia e Benchmark de Não Recorrência | Consignado

## Objetivo

Este notebook é a terceira etapa do estudo de não recorrência de Consignado.

Depois de identificar as lojas que **produziram até o 14º DU do mês anterior e ainda não produziram até o mesmo DU do mês atual**, e de estudar seu comportamento histórico, agora queremos responder:

> **A perda parece estar associada à loja, ao perfil da praça, ao território ou à estrutura comercial?**

### Fontes

**Produção**
`TESTE..PRODUCAO_DIA_UTIL_BE`

**Hierarquia / cadastro da loja**
`DATALAKE..DL_BRADESCO_EXPRESSO`

Campos esperados:
- `CHAVE_LOJA`
- `CD_MUNIC`
- `MUNICIPIO`
- `UF`
- `DESC_GERENCIA_AREA`
- `DESC_COORDENACAO`
- `DESC_SUPERVISAO`

**População**
`IBGE..IBGE_POP`

Chave:
`CD_MUNIC = COD_UN_REG`

### Estrutura analítica

O estudo seguirá:

**Loja → perfil comparável → município/UF → Gerência de Gestão → GC III → GC**

Não analisaremos apenas quantidade absoluta de perdas. O principal indicador será a **taxa de não recorrência entre lojas elegíveis**.


In [ ]:
# 1. Bibliotecas e parâmetros

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from db import read_sql

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PERIODO_ANTERIOR = 202607
PERIODO_ATUAL = 202608
DU_CORTE = 14

PRODUTOS = {
    "CONTABIL": ("QTD_TRX", "QTD_TRX_ACUM"),
    "CRED_TOTAL": ("CRED_TOTAL", "CRED_TOTAL_ACUM"),
    "CREDITO_JORNADA": ("CREDITO_JORNADA", "CREDITO_JORNADA_ACUM"),
    "PARCELADO": ("VLR_CREDITO_PARCEL", "VLR_CREDITO_PARCEL_ACUM"),
    "CONSIG": ("VLR_CONSIG", "VLR_CONSIG_ACUM"),
    "LIME": ("VLR_LIME", "VLR_LIME_ACUM"),
    "CONTAS": ("CONTAS", "CONTAS_ACUM"),
    "CARTAO_CREDITO": ("QTD_CARTAO_CONTRATADO", "QTD_CARTAO_CONTRATADO_ACUM"),
    "SEGUROS": ("SEG_TOTAL", "SEG_TOTAL_ACUM"),
}

# Evita interpretar grupos muito pequenos como evidência territorial forte.
MIN_BASE_GRUPO = 10

print(
    f"Estudo territorial: {PERIODO_ANTERIOR} x {PERIODO_ATUAL} "
    f"| até {DU_CORTE}º DU"
)


## 2. Extração integrada

A consulta abaixo monta uma base única com produção, hierarquia e população.

> **Atenção:** confirme o nome da coluna de população em `IBGE..IBGE_POP`.  
> No código abaixo foi utilizado `POPULACAO`. Ajuste se necessário.

Também é importante validar se `DATALAKE..DL_BRADESCO_EXPRESSO` possui apenas um registro válido por `CHAVE_LOJA`. Caso existam múltiplos registros, será necessário definir a regra correta de vigência antes do `JOIN`.


In [ ]:
# 2. Extração integrada (conexão em db.py)

query = f"""
SELECT
    A.*,

    B.CD_MUNIC,
    B.MUNICIPIO,
    B.UF,
    B.DESC_GERENCIA_AREA,
    B.DESC_COORDENACAO,
    B.DESC_SUPERVISAO,

    C.POPULACAO

FROM TESTE..PRODUCAO_DIA_UTIL_BE A

LEFT JOIN DATALAKE..DL_BRADESCO_EXPRESSO B
    ON A.CHAVE_LOJA = B.CHAVE_LOJA

LEFT JOIN IBGE..IBGE_POP C
    ON B.CD_MUNIC = C.COD_UN_REG

WHERE A.PERIODO IN ({PERIODO_ANTERIOR}, {PERIODO_ATUAL})
  AND A.QTD_DIA_UTIL_MES <= {DU_CORTE}
"""

print(query)

df = read_sql(query)


## 3. Auditoria dos JOINs

Antes da análise territorial, precisamos garantir que os `LEFT JOINs` não multiplicaram linhas.

A granularidade esperada continua sendo:

`CHAVE_LOJA + PERIODO + QTD_DIA_UTIL_MES`


In [ ]:
def auditar_integracao(df):
    chave = ["CHAVE_LOJA", "PERIODO", "QTD_DIA_UTIL_MES"]

    print("Linhas:", f"{len(df):,}")
    print("Lojas:", f"{df['CHAVE_LOJA'].nunique():,}")

    duplicadas = df.duplicated(chave, keep=False)
    print("Linhas duplicadas na granularidade:", f"{duplicadas.sum():,}")

    campos = [
        "CD_MUNIC", "MUNICIPIO", "UF", "POPULACAO",
        "DESC_GERENCIA_AREA", "DESC_COORDENACAO", "DESC_SUPERVISAO"
    ]

    print("\nCobertura cadastral:")
    for c in campos:
        if c in df.columns:
            pct = df[c].notna().mean() * 100
            print(f"{c:<25} {pct:>6.2f}% preenchido")

    if duplicadas.any():
        print(
            "\nATENÇÃO: o JOIN multiplicou registros ou a fonte original "
            "já possui duplicidades. Corrija antes de continuar."
        )

# auditar_integracao(df)


## 4. Padronização territorial

Criamos faixas populacionais para não comparar diretamente mercados estruturalmente muito diferentes.

Faixas iniciais:

- `< 30 mil`
- `30–50 mil`
- `50–100 mil`
- `100–300 mil`
- `> 300 mil`

Essas faixas podem ser ajustadas posteriormente.


In [ ]:
def preparar_territorio(df):
    out = df.copy()

    out["CD_MUNIC"] = out["CD_MUNIC"].astype("string")
    out["UF"] = out["UF"].astype("string").str.strip().str.upper()
    out["MUNICIPIO"] = out["MUNICIPIO"].astype("string").str.strip()

    out["POPULACAO"] = pd.to_numeric(
        out["POPULACAO"], errors="coerce"
    )

    out["FAIXA_POP"] = pd.cut(
        out["POPULACAO"],
        bins=[-1, 29999, 49999, 99999, 299999, np.inf],
        labels=[
            "<30 mil",
            "30–50 mil",
            "50–100 mil",
            "100–300 mil",
            ">300 mil"
        ]
    )

    return out

# df = preparar_territorio(df)


## 5. Reconstrução da coorte no 14º DU

O denominador correto para taxa de perda é:

> **lojas que haviam produzido Consignado até o 14º DU do mês anterior**

Dentro desse universo:

- `MANTEVE`
- `PERDEU`

Também mantemos `ENTROU` para estudar recomposição da base.


In [ ]:
def snapshot(df, periodo, du):
    base = df[
        (df["PERIODO"] == periodo) &
        (df["QTD_DIA_UTIL_MES"] <= du)
    ].copy()

    base = (
        base.sort_values(["CHAVE_LOJA", "QTD_DIA_UTIL_MES"])
            .groupby("CHAVE_LOJA", as_index=False)
            .tail(1)
    )

    return base


def construir_coorte_territorial(df):
    ant = snapshot(df, PERIODO_ANTERIOR, DU_CORTE)
    atual = snapshot(df, PERIODO_ATUAL, DU_CORTE)

    cadastro_cols = [
        "CHAVE_LOJA", "CD_MUNIC", "MUNICIPIO", "UF", "POPULACAO",
        "FAIXA_POP", "DESC_GERENCIA_AREA",
        "DESC_COORDENACAO", "DESC_SUPERVISAO"
    ]
    cadastro_cols = [c for c in cadastro_cols if c in atual.columns]

    cadastro = atual[cadastro_cols].drop_duplicates("CHAVE_LOJA")

    a = ant[["CHAVE_LOJA", "QTD_CONSIG_ACUM"]].rename(
        columns={"QTD_CONSIG_ACUM": "QTD_CONSIG_ACUM_ANT"}
    )
    b = atual[["CHAVE_LOJA", "QTD_CONSIG_ACUM"]].rename(
        columns={"QTD_CONSIG_ACUM": "QTD_CONSIG_ACUM_ATUAL"}
    )

    base = a.merge(b, on="CHAVE_LOJA", how="outer").fillna(0)
    base = base.merge(cadastro, on="CHAVE_LOJA", how="left")

    base["PROD_ANT"] = (base["QTD_CONSIG_ACUM_ANT"] > 0).astype(int)
    base["PROD_ATUAL"] = (base["QTD_CONSIG_ACUM_ATUAL"] > 0).astype(int)

    base["STATUS_CONSIG"] = np.select(
        [
            (base["PROD_ANT"] == 1) & (base["PROD_ATUAL"] == 1),
            (base["PROD_ANT"] == 1) & (base["PROD_ATUAL"] == 0),
            (base["PROD_ANT"] == 0) & (base["PROD_ATUAL"] == 1),
        ],
        ["MANTEVE", "PERDEU", "ENTROU"],
        default="INATIVA"
    )

    base["FLAG_PERDA"] = (base["STATUS_CONSIG"] == "PERDEU").astype(int)
    base["FLAG_MANTEVE"] = (base["STATUS_CONSIG"] == "MANTEVE").astype(int)
    base["FLAG_ENTROU"] = (base["STATUS_CONSIG"] == "ENTROU").astype(int)

    return base

# coorte = construir_coorte_territorial(df)
# display(coorte["STATUS_CONSIG"].value_counts().to_frame("LOJAS"))


In [ ]:
# CHECKPOINT

# qtd_perdas = (coorte["STATUS_CONSIG"] == "PERDEU").sum()
# print("Lojas classificadas como PERDEU:", qtd_perdas)

# if qtd_perdas != 502:
#     print(
#         "ATENÇÃO: esperado 502. Verifique JOIN, cadastro, granularidade "
#         "ou disponibilidade do 14º DU."
#     )


# 6. Taxa nacional de não recorrência

Essa taxa será o primeiro benchmark:

`PERDEU / (PERDEU + MANTEVE)`

Ela representa a parcela das lojas que já tinham produzido até o mesmo DU do mês anterior e ainda não retornaram.


In [ ]:
def taxa_nacional(coorte):
    elegiveis = coorte[coorte["PROD_ANT"] == 1]

    base = elegiveis["CHAVE_LOJA"].nunique()
    perdas = elegiveis.loc[
        elegiveis["STATUS_CONSIG"] == "PERDEU", "CHAVE_LOJA"
    ].nunique()

    taxa = perdas / base * 100 if base else np.nan

    print(f"Base elegível: {base:,}")
    print(f"Perdas: {perdas:,}")
    print(f"Taxa nacional de não recorrência: {taxa:.2f}%")

    return taxa

# TX_NACIONAL = taxa_nacional(coorte)


# 7. Função geral de benchmark

A mesma função será usada para:

- Gerência de Gestão;
- GC III;
- GC;
- UF;
- município;
- faixa populacional.

Além da taxa, calculamos diferença em pontos percentuais contra o benchmark nacional.


In [ ]:
def benchmark_dimensao(coorte, dimensao, taxa_nacional=None, min_base=1):
    elegiveis = coorte[coorte["PROD_ANT"] == 1].copy()

    resumo = (
        elegiveis.groupby(dimensao, dropna=False)
        .agg(
            BASE_ELEGIVEL=("CHAVE_LOJA", "nunique"),
            PERDAS=("FLAG_PERDA", "sum"),
            MANTIDAS=("FLAG_MANTEVE", "sum")
        )
        .reset_index()
    )

    resumo["TX_NAO_RECORRENCIA"] = (
        resumo["PERDAS"] / resumo["BASE_ELEGIVEL"] * 100
    )

    if taxa_nacional is not None:
        resumo["DELTA_VS_BRASIL_PP"] = (
            resumo["TX_NAO_RECORRENCIA"] - taxa_nacional
        )

    resumo["BASE_CONFIAVEL"] = resumo["BASE_ELEGIVEL"] >= min_base

    return resumo.sort_values(
        ["BASE_CONFIAVEL", "TX_NAO_RECORRENCIA", "BASE_ELEGIVEL"],
        ascending=[False, False, False]
    )

# gerencia = benchmark_dimensao(
#     coorte, "DESC_GERENCIA_AREA", TX_NACIONAL, MIN_BASE_GRUPO
# )
# display(gerencia)


## 8. Visões hierárquicas

Aqui começamos a responder:

> **Onde a não recorrência está acima do esperado?**

Importante: não interpretar automaticamente uma taxa alta como baixa performance comercial. Primeiro precisamos controlar perfil territorial e tamanho da carteira.


In [ ]:
# visoes = {}

# for dimensao in [
#     "DESC_GERENCIA_AREA",
#     "DESC_COORDENACAO",
#     "DESC_SUPERVISAO",
#     "UF",
#     "FAIXA_POP",
#     "MUNICIPIO"
# ]:
#     if dimensao in coorte.columns:
#         visoes[dimensao] = benchmark_dimensao(
#             coorte,
#             dimensao,
#             taxa_nacional=TX_NACIONAL,
#             min_base=MIN_BASE_GRUPO
#         )

# display(visoes["DESC_GERENCIA_AREA"])
# display(visoes["DESC_COORDENACAO"].head(20))
# display(visoes["DESC_SUPERVISAO"].head(30))


# 9. Visual — taxa por Gerência de Gestão

O gráfico compara **taxa**, não quantidade absoluta.

A linha horizontal representa o benchmark nacional.


In [ ]:
def plot_taxa_horizontal(tabela, dimensao, taxa_nacional, top_n=None):
    dados = tabela[tabela["BASE_CONFIAVEL"]].copy()

    if top_n:
        dados = dados.head(top_n)

    dados = dados.sort_values("TX_NAO_RECORRENCIA")

    fig, ax = plt.subplots(figsize=(11, max(4, len(dados) * 0.45)))

    ax.barh(
        dados[dimensao].astype(str),
        dados["TX_NAO_RECORRENCIA"]
    )

    ax.axvline(
        taxa_nacional,
        linestyle="--",
        linewidth=1.5,
        label=f"Brasil: {taxa_nacional:.1f}%"
    )

    for i, valor in enumerate(dados["TX_NAO_RECORRENCIA"]):
        ax.text(valor + 0.4, i, f"{valor:.1f}%", va="center")

    ax.set_title(
        f"Taxa de não recorrência por {dimensao}",
        loc="left",
        fontweight="bold"
    )
    ax.set_xlabel("% das lojas elegíveis")
    ax.set_ylabel("")
    ax.legend(frameon=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()

# plot_taxa_horizontal(
#     visoes["DESC_GERENCIA_AREA"],
#     "DESC_GERENCIA_AREA",
#     TX_NACIONAL
# )


# 10. Perfil populacional

Antes de comparar estruturas comerciais, precisamos saber se suas carteiras possuem perfis territoriais diferentes.

Uma Gerência com forte concentração em municípios pequenos pode apresentar comportamento estruturalmente diferente de outra concentrada em grandes centros.


In [ ]:
def perfil_carteira(coorte, estrutura="DESC_GERENCIA_AREA"):
    elegiveis = coorte[coorte["PROD_ANT"] == 1].copy()

    tabela = pd.crosstab(
        elegiveis[estrutura],
        elegiveis["FAIXA_POP"],
        normalize="index"
    ).mul(100)

    return tabela

# perfil_gerencia = perfil_carteira(coorte, "DESC_GERENCIA_AREA")
# display(perfil_gerencia.round(1))


# 11. Benchmark por perfil territorial

Agora comparamos lojas dentro de contextos mais semelhantes.

Primeiro peer group sugerido:

`UF + FAIXA_POP`

Depois podemos evoluir para:

`UF + FAIXA_POP + perfil histórico + porte comercial`

O objetivo é evitar comparar diretamente uma loja de município pequeno com uma loja de grande centro.


In [ ]:
def criar_peer_group(coorte):
    out = coorte.copy()

    out["PEER_GROUP"] = (
        out["UF"].fillna("SEM_UF").astype(str)
        + " | "
        + out["FAIXA_POP"].astype("string").fillna("SEM_FAIXA")
    )

    return out

# coorte = criar_peer_group(coorte)

# peer_benchmark = benchmark_dimensao(
#     coorte,
#     "PEER_GROUP",
#     taxa_nacional=TX_NACIONAL,
#     min_base=MIN_BASE_GRUPO
# )

# display(peer_benchmark.head(30))


# 12. Benchmark individual da loja

Para cada loja elegível, anexamos:

- taxa nacional;
- taxa da UF;
- taxa da faixa populacional;
- taxa do peer group;
- taxa da Gerência;
- taxa do GC III;
- taxa do GC.

Assim uma loja perdida deixa de ser apenas `PERDEU = 1`; ela passa a carregar seu contexto.


In [ ]:
def anexar_benchmark(base, tabela, dimensao, prefixo):
    cols = [
        dimensao,
        "BASE_ELEGIVEL",
        "TX_NAO_RECORRENCIA"
    ]

    aux = tabela[cols].copy().rename(columns={
        "BASE_ELEGIVEL": f"BASE_{prefixo}",
        "TX_NAO_RECORRENCIA": f"TX_PERDA_{prefixo}"
    })

    return base.merge(aux, on=dimensao, how="left")


# Exemplo de montagem:
#
# coorte_contexto = coorte.copy()
# coorte_contexto["TX_PERDA_BRASIL"] = TX_NACIONAL
#
# for dimensao, prefixo in [
#     ("UF", "UF"),
#     ("FAIXA_POP", "FAIXA_POP"),
#     ("PEER_GROUP", "PEER"),
#     ("DESC_GERENCIA_AREA", "GERENCIA"),
#     ("DESC_COORDENACAO", "GCIII"),
#     ("DESC_SUPERVISAO", "GC")
# ]:
#     if dimensao in visoes:
#         tabela = visoes[dimensao]
#     else:
#         tabela = benchmark_dimensao(
#             coorte, dimensao, TX_NACIONAL, MIN_BASE_GRUPO
#         )
#
#     coorte_contexto = anexar_benchmark(
#         coorte_contexto, tabela, dimensao, prefixo
#     )


# 13. Índice de efeito territorial

Uma loja perdida dentro de um peer group em que **muitas lojas também perderam** pode refletir um fenômeno de mercado.

Uma loja perdida dentro de um peer group com **baixa taxa de perda** parece mais idiossincrática.

Criamos uma leitura inicial:

- `EFEITO TERRITORIAL FORTE`
- `EFEITO TERRITORIAL MODERADO`
- `FORA DO PADRÃO LOCAL`

Essa classificação é uma hipótese, não causalidade comprovada.


In [ ]:
def classificar_contexto_territorial(row, tx_nacional):
    tx_peer = row.get("TX_PERDA_PEER", np.nan)
    base_peer = row.get("BASE_PEER", 0)

    if pd.isna(tx_peer) or base_peer < MIN_BASE_GRUPO:
        return "SEM BASE LOCAL SUFICIENTE"

    delta = tx_peer - tx_nacional

    if delta >= 15:
        return "EFEITO TERRITORIAL FORTE"

    if delta >= 5:
        return "EFEITO TERRITORIAL MODERADO"

    if delta <= -5:
        return "FORA DO PADRÃO LOCAL"

    return "PRÓXIMO AO PADRÃO NACIONAL"

# perdas_contexto = coorte_contexto.query(
#     "STATUS_CONSIG == 'PERDEU'"
# ).copy()
#
# perdas_contexto["CONTEXTO_TERRITORIAL"] = perdas_contexto.apply(
#     lambda r: classificar_contexto_territorial(r, TX_NACIONAL),
#     axis=1
# )
#
# display(
#     perdas_contexto["CONTEXTO_TERRITORIAL"]
#     .value_counts()
#     .to_frame("LOJAS")
# )


# 14. Praça × Estrutura comercial

Este bloco procura situações especialmente interessantes:

### A. Mercado ruim + GC ruim
Pode haver forte componente territorial.

### B. Mercado saudável + GC ruim
Merece investigação da carteira/atuação.

### C. Mercado ruim + GC saudável
A estrutura parece estar performando relativamente melhor que o contexto.

### D. Mercado saudável + GC saudável
Perdas individuais tendem a ser mais específicas da loja.

Para reduzir distorções, apenas grupos com base mínima devem ser interpretados.


In [ ]:
def matriz_efeito_estrutura(perdas_contexto, tx_nacional):
    out = perdas_contexto.copy()

    out["PEER_ACIMA_BRASIL"] = (
        out["TX_PERDA_PEER"] > tx_nacional + 5
    )

    out["GC_ACIMA_BRASIL"] = (
        out["TX_PERDA_GC"] > tx_nacional + 5
    )

    out["LEITURA_ESTRUTURAL"] = np.select(
        [
            out["PEER_ACIMA_BRASIL"] & out["GC_ACIMA_BRASIL"],
            ~out["PEER_ACIMA_BRASIL"] & out["GC_ACIMA_BRASIL"],
            out["PEER_ACIMA_BRASIL"] & ~out["GC_ACIMA_BRASIL"],
        ],
        [
            "MERCADO E CARTEIRA SOB PRESSÃO",
            "CARTEIRA PIOR QUE O CONTEXTO",
            "CARTEIRA RESILIENTE EM MERCADO SOB PRESSÃO",
        ],
        default="PERDA MAIS ESPECÍFICA / CONTEXTO SAUDÁVEL"
    )

    return out

# perdas_contexto = matriz_efeito_estrutura(
#     perdas_contexto, TX_NACIONAL
# )
#
# display(
#     perdas_contexto["LEITURA_ESTRUTURAL"]
#     .value_counts()
#     .to_frame("LOJAS")
# )


# 15. Concentração das perdas

Quantidade ainda importa, desde que acompanhada da taxa.

Uma estrutura pode ter:

- muitas perdas porque possui carteira grande;
- poucas perdas, mas taxa extremamente alta;
- alta taxa e alto volume simultaneamente.

Esse último quadrante merece atenção especial.


In [ ]:
def classificar_prioridade_estrutura(tabela):
    out = tabela.copy()

    mediana_perdas = out.loc[
        out["BASE_CONFIAVEL"], "PERDAS"
    ].median()

    mediana_taxa = out.loc[
        out["BASE_CONFIAVEL"], "TX_NAO_RECORRENCIA"
    ].median()

    out["PRIORIDADE_ESTRUTURA"] = np.select(
        [
            (out["PERDAS"] >= mediana_perdas) &
            (out["TX_NAO_RECORRENCIA"] >= mediana_taxa),

            (out["PERDAS"] < mediana_perdas) &
            (out["TX_NAO_RECORRENCIA"] >= mediana_taxa),

            (out["PERDAS"] >= mediana_perdas) &
            (out["TX_NAO_RECORRENCIA"] < mediana_taxa),
        ],
        [
            "ALTO VOLUME + ALTA TAXA",
            "BAIXO VOLUME + ALTA TAXA",
            "ALTO VOLUME + TAXA CONTROLADA",
        ],
        default="BAIXO VOLUME + TAXA CONTROLADA"
    )

    return out

# gc_prioridade = classificar_prioridade_estrutura(
#     visoes["DESC_SUPERVISAO"]
# )
# display(gc_prioridade.head(30))


# 16. Gráfico Volume × Taxa

Esse gráfico ajuda a evitar rankings simplistas.

- eixo X = base elegível;
- eixo Y = taxa de não recorrência;
- tamanho do ponto = quantidade de perdas.

Estruturas no alto e à direita merecem investigação.


In [ ]:
def plot_volume_taxa(tabela, dimensao, min_base=MIN_BASE_GRUPO):
    dados = tabela[tabela["BASE_ELEGIVEL"] >= min_base].copy()

    fig, ax = plt.subplots(figsize=(11, 6))

    tamanho = np.maximum(dados["PERDAS"].to_numpy(), 1) * 8

    ax.scatter(
        dados["BASE_ELEGIVEL"],
        dados["TX_NAO_RECORRENCIA"],
        s=tamanho,
        alpha=0.65
    )

    for _, r in dados.iterrows():
        ax.annotate(
            str(r[dimensao]),
            (r["BASE_ELEGIVEL"], r["TX_NAO_RECORRENCIA"]),
            xytext=(5, 4),
            textcoords="offset points",
            fontsize=8
        )

    ax.set_title(
        f"Volume da carteira × taxa de não recorrência — {dimensao}",
        loc="left",
        fontweight="bold"
    )
    ax.set_xlabel("Lojas elegíveis")
    ax.set_ylabel("Taxa de não recorrência (%)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()

# plot_volume_taxa(
#     visoes["DESC_COORDENACAO"],
#     "DESC_COORDENACAO"
# )


# 17. Insight por Gerência de Gestão

A função abaixo produz um resumo executivo por Gerência, combinando:

- base elegível;
- perdas;
- taxa;
- comparação com Brasil;
- concentração por faixa populacional.

É uma primeira ponte para o PowerPoint.


In [ ]:
def resumo_executivo_gerencias(coorte, taxa_nacional):
    ger = benchmark_dimensao(
        coorte,
        "DESC_GERENCIA_AREA",
        taxa_nacional,
        min_base=1
    )

    elegiveis = coorte[coorte["PROD_ANT"] == 1].copy()

    faixa_dominante = (
        elegiveis.groupby(["DESC_GERENCIA_AREA", "FAIXA_POP"], observed=True)
        .size()
        .rename("QTD")
        .reset_index()
        .sort_values(
            ["DESC_GERENCIA_AREA", "QTD"],
            ascending=[True, False]
        )
        .drop_duplicates("DESC_GERENCIA_AREA")
        [["DESC_GERENCIA_AREA", "FAIXA_POP"]]
        .rename(columns={"FAIXA_POP": "PERFIL_TERRITORIAL_DOMINANTE"})
    )

    ger = ger.merge(
        faixa_dominante,
        on="DESC_GERENCIA_AREA",
        how="left"
    )

    ger["LEITURA"] = np.select(
        [
            ger["DELTA_VS_BRASIL_PP"] >= 5,
            ger["DELTA_VS_BRASIL_PP"] <= -5
        ],
        [
            "ACIMA DO PADRÃO NACIONAL",
            "ABAIXO DO PADRÃO NACIONAL"
        ],
        default="PRÓXIMO AO PADRÃO NACIONAL"
    )

    return ger

# resumo_gerencias = resumo_executivo_gerencias(
#     coorte, TX_NACIONAL
# )
# display(resumo_gerencias)


# 18. Integração com o Notebook 02

O maior ganho virá ao juntar esta base territorial com o arquivo gerado no Notebook 02.

Assim cada uma das perdas poderá conter simultaneamente:

### Histórico da loja
- recorrência;
- primeiro DU habitual;
- score de anormalidade;
- potencial histórico.

### Comportamento atual
- outros produtos;
- possível mudança de mix;
- deterioração ampla.

### Contexto
- taxa da praça;
- taxa do peer group;
- taxa do GC;
- taxa do GC III;
- taxa da Gerência.

Isso permite construir uma leitura muito mais rica:

> **Ruptura forte + mercado saudável + outros produtos preservados = oportunidade prioritária de retomada específica de Consignado.**


In [ ]:
# Exemplo caso tenha exportado o Notebook 02:

# perfil_hist = pd.read_excel(
#     "outputs/02_perfil_historico_perdas_consignado.xlsx"
# )

# perdas_final = perdas_contexto.merge(
#     perfil_hist,
#     on="CHAVE_LOJA",
#     how="left",
#     suffixes=("", "_HIST")
# )


# 19. Score contextual de oportunidade

Após integrar o Notebook 02, podemos combinar:

- anormalidade histórica;
- potencial;
- preservação dos demais produtos;
- comportamento do peer group;
- contexto da estrutura comercial.

O exemplo abaixo é apenas uma estrutura inicial. Os pesos devem ser calibrados após observar os dados reais.


In [ ]:
def normalizar_0_100(s):
    s = pd.to_numeric(s, errors="coerce")
    mn, mx = s.min(), s.max()

    if pd.isna(mn) or pd.isna(mx) or mn == mx:
        return pd.Series(0, index=s.index, dtype=float)

    return ((s - mn) / (mx - mn) * 100).clip(0, 100)


def score_contextual(base):
    out = base.copy()

    # Quanto MENOR a taxa de perda do peer,
    # mais específica/anormal é a perda da loja.
    out["SCORE_ESPECIFICIDADE_LOCAL"] = (
        100 - out["TX_PERDA_PEER"].fillna(out["TX_PERDA_BRASIL"])
    ).clip(0, 100)

    if "SCORE_ANORMALIDADE" in out.columns:
        anormalidade = out["SCORE_ANORMALIDADE"].fillna(0)
    else:
        anormalidade = 0

    if "VLR_CONSIG_MEDIA_6M" in out.columns:
        potencial = normalizar_0_100(
            np.log1p(out["VLR_CONSIG_MEDIA_6M"].fillna(0))
        )
    else:
        potencial = 0

    out["SCORE_OPORTUNIDADE_CONTEXTUAL"] = (
        0.45 * anormalidade +
        0.30 * potencial +
        0.25 * out["SCORE_ESPECIFICIDADE_LOCAL"]
    ).round(1)

    return out

# perdas_final = score_contextual(perdas_final)


# 20. Perguntas que o Notebook 03 deve responder

Ao final da execução, procure responder:

1. **Qual é a taxa nacional de não recorrência?**
2. **Quais Gerências de Gestão estão significativamente acima dela?**
3. **Quais GC III e GCs concentram alta taxa + alto volume?**
4. **O comportamento muda por faixa populacional?**
5. **Quais UFs/municípios apresentam fenômeno coletivo?**
6. **Quais lojas são perdas isoladas dentro de mercados saudáveis?**
7. **Quais carteiras estão piores do que o perfil territorial que atendem?**
8. **Quais estruturas parecem resilientes mesmo em mercados sob pressão?**
9. **Após integrar o Notebook 02, onde estão as rupturas historicamente anormais e de alto potencial?**

A intenção é sair de:

**“502 lojas não retornaram”**

para:

**“Estas são as perdas que parecem estruturais, estas parecem territoriais e estas são oportunidades específicas de retomada.”**


# 21. Exportações sugeridas

As saídas serão úteis tanto para aprofundamento analítico quanto para construção da apresentação executiva.


In [ ]:
# Path("outputs").mkdir(exist_ok=True)

# coorte.to_excel(
#     "outputs/03_coorte_territorial.xlsx",
#     index=False
# )

# resumo_gerencias.to_excel(
#     "outputs/03_resumo_gerencias.xlsx",
#     index=False
# )

# peer_benchmark.to_excel(
#     "outputs/03_benchmark_peer_groups.xlsx",
#     index=False
# )

# perdas_contexto.to_excel(
#     "outputs/03_perdas_contextualizadas.xlsx",
#     index=False
# )

print("Notebook 03 estruturado e pronto para execução.")


## Próxima etapa sugerida

Depois de rodar este notebook, **não avançar automaticamente para ML**.

Primeiro devemos olhar os resultados reais e verificar quais diferenças territoriais aparecem.

A partir disso, o Notebook 04 pode ser desenhado especificamente para os fenômenos encontrados — por exemplo:

**explicabilidade estatística + drivers por região + priorização comercial**, em vez de impor um único modelo nacional sobre mercados com comportamentos diferentes.
